In [0]:
catalog = "workspace"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.worldbank")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.worldbank.raw_files")
spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA worldbank")

vol_path = f"/Volumes/{catalog}/worldbank/raw_files"
print("ok, volume:", vol_path)

ok, volume: /Volumes/workspace/worldbank/raw_files


In [0]:
import requests

url = "https://search.worldbank.org/api/v3/wds"
params = {"format": "json", "rows": 2, "qterm": "climate change"}

resp = requests.get(url, params=params, timeout=30)
print("status:", resp.status_code)

data = resp.json()
print("total available:", data.get("total"))
print("sample keys:", list(data["documents"].keys())[:5])

status: 200
total available: 265411
sample keys: ['D25933072', 'D32995746', 'facets']


In [0]:
import requests, json, time

url = "https://search.worldbank.org/api/v3/wds"
qterm = "climate change"
page_size = 100
max_records = 4000

records = []
offset = 0

while len(records) < max_records:
    params = {"format": "json", "qterm": qterm, "rows": page_size, "os": offset}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    docs = r.json().get("documents", {})
    page = [v for k, v in docs.items() if k != "facets"]   # выкидываем служебный ключ
    if not page:
        break
    records.extend(page)
    offset += len(page)
    print(f"  fetched {len(records)}")
    time.sleep(0.3)   # вежливая пауза, чтобы не долбить API

records = records[:max_records]

out_file = f"{vol_path}/worldbank_docs.json"
with open(out_file, "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

print("done:", len(records), "records ->", out_file)

  fetched 100
  fetched 200
  fetched 300
  fetched 400
  fetched 500
  fetched 600
  fetched 700
  fetched 800
  fetched 900
  fetched 1000
  fetched 1100
  fetched 1200
  fetched 1300
  fetched 1400
  fetched 1500
  fetched 1600
  fetched 1700
  fetched 1800
  fetched 1900
  fetched 2000
  fetched 2100
  fetched 2200
  fetched 2300
  fetched 2400
  fetched 2500
  fetched 2600
  fetched 2700
  fetched 2800
  fetched 2900
  fetched 3000
  fetched 3100
  fetched 3200
  fetched 3300
  fetched 3400
  fetched 3500
  fetched 3600
  fetched 3700
  fetched 3800
  fetched 3900
  fetched 4000
done: 4000 records -> /Volumes/workspace/worldbank/raw_files/worldbank_docs.json


In [0]:
from pyspark.sql import functions as F

bronze = (
    spark.read.text(f"{vol_path}/worldbank_docs.json")
    .withColumnRenamed("value", "raw")
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit("worldbank_wds_api"))
)

bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_docs")

print("bronze rows:", spark.table("bronze_docs").count())
display(spark.table("bronze_docs").limit(3))

bronze rows: 4000


raw ingested_at source {"id": "25933072", "admreg": "Other", "authors": {"0": {"author": "Bangalore, Mukund Ram"}, "1": {"author": "Hallegatte, Stephane"}, "2": {"author": "Bangalore, Mook"}, "3": {"author": "Bonzanigo, Laura"}, "4": {"author": "Fay, Marianne"}, "5": {"author": "Kane, Tamaro"}, "6": {"author": "Narloch, Ulf Gerrit"}, "7": {"author": "Rozenberg, Julie"}, "8": {"author": "Treguer, David Olivier"}, "9": {"author": "Vogt-Schilb, Adrien Camille"}}, "count": "World", "docna": {"0": {"docna": "Shock waves : managing the impacts of climate change on poverty"}}, "docty": "Publication", "owner": "GCC - Research and Advisory Unit (GCCRA)", "projn": "1W-Climate Change and Poverty -- P149919", "subsc": "FY17 - Other Energy and Extractives,FY17 - Other Water Supply, Sanitation and Waste Management,FY17 - Health,FY17 - Social Protection,FY17 - Other Agriculture, Fishing and Forestry", "theme": "FY17 - Environmental Health and Pollution Management,FY17 - Climate change,FY17 - Economic Growth and Planning,FY17 - Air quality management,FY17 - Green Growth,FY17 - Inclusive Growth,FY17 - Soil Pollution,FY17 - Social Protection,FY17 - Urban Development,FY17 - Urban Infrastructure and Service Delivery,FY17 - Mitigation,FY17 - Social Safety Nets,FY17 - Water Pollution,FY17 - Structural Transformation and Economic Diversification", "prdln": "Economic and Sector Work", "seccl": "Public", "lang": "English", "entityids": {"entityid": "090224b08418b822_1_0"}, "majtheme": "FY17 - Environment and Natural Resource Management,FY17 - Urban and Rural Development,FY17 - Economic Policy,FY17 - Social Development and Protection", "subtopic": "Cholera,Climate Change and Environment,Climate Change and Health,Communicable Diseases,Food Security,Health Care Services Industry,Inequality,Leprosy,Malaria,Science of Climate Change", "teratopic": "Agriculture,Environment,Health, Nutrition and Population,Industry,Poverty Reduction,Science and Technology Development", "sectr": {"0": {"sector": "FY17 - Water, Sanitation and Waste Management"}, "1": {"sector": "FY17 - Social Protection"}, "2": {"sector": "FY17 - Agriculture, Fishing and Forestry"}, "3": {"sector": "FY17 - Energy and Extractives"}, "4": {"sector": "FY17 - Health"}}, "repnb": "100758", "docdt": "2016-01-01T05:00:00Z", "datestored": "2016-08-05T04:00:00Z", "keywd": {"0": {"keywd": "Demographic and Health Survey"}, "1": {"keywd": "danish international development"}, "2": {"keywd": "impact of climate change"}, "3": {"keywd": "vulnerability to climate change"}, "4": {"keywd": "threat of climate change"}, "5": {"keywd": "United Nations Environment Programme"}, "6": {"keywd": "cost of health care"}, "7": {"keywd": "Social Safety Nets"}, "8": {"keywd": "impact on poverty"}, "9": {"keywd": "threats to health"}, "10": {"keywd": "climate change impact"}, "11": {"keywd": "people in poverty"}, "12": {"keywd": "social protection scheme"}, "13": {"keywd": "fossil fuel subsidy"}, "14": {"keywd": "extreme weather event"}, "15": {"keywd": "availability of land"}, "16": {"keywd": "tropical forest landscapes"}, "17": {"keywd": "climate change mitigation"}, "18": {"keywd": "food price"}, "19": {"keywd": "crop yield loss"}, "20": {"keywd": "global carbon emission"}, "21": {"keywd": "post-traumatic stress disorder"}, "22": {"keywd": "expenditure on food"}, "23": {"keywd": "annual monthly temperature"}, "24": {"keywd": "purchasing power parity"}, "25": {"keywd": "child mortality rate"}, "26": {"keywd": "rapid population growth"}, "27": {"keywd": "poor urban dwellers"}, "28": {"keywd": "incidence of malaria"}, "29": {"keywd": "global mean temperature"}, "30": {"keywd": "risk financing"}, "31": {"keywd": "tons of carbon"}, "32": {"keywd": "school feeding scheme"}, "33": {"keywd": "health care system"}, "34": {"keywd": "land and housing"}, "35": {"keywd": "eradication of poverty"}, "36": {"keywd": "Migration and Remittances"}, "37": {"keywd": "sea level rise"}, "38": {"keywd": "coastal flood risk"}, "39": {"keywd": "lac

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

# явная схема: берём только нужные поля, остальное Spark игнорирует
schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("display_title", T.StringType()),
    T.StructField("docdt", T.StringType()),
    T.StructField("count", T.StringType()),
    T.StructField("admreg", T.StringType()),
    T.StructField("docty", T.StringType()),
    T.StructField("lang", T.StringType()),
    T.StructField("majdocty", T.StringType()),
])

bronze = spark.table("bronze_docs")

parsed = bronze.withColumn("j", F.from_json("raw", schema)).select("j.*", "ingested_at")

checked = (
    parsed
    .withColumn("doc_date", F.to_date(F.col("docdt")))
    .withColumn("doc_year", F.year("doc_date"))
    .withColumn(
        "reject_reason",
        F.when(F.col("id").isNull(), "missing_id")
         .when(F.col("doc_date").isNull(), "bad_date")
         .otherwise(None)
    )
)

# карантин: записи без id или с непарсящейся датой
quarantine = (
    checked.filter(F.col("reject_reason").isNotNull())
    .select("id", "display_title", "docdt", "count", "reject_reason")
    .withColumn("quarantined_at", F.current_timestamp())
)

# хорошие записи + дедуп по id (оставляем самую свежезагруженную)
good = checked.filter(F.col("reject_reason").isNull()).drop("reject_reason", "docdt")

key = Window.partitionBy("id").orderBy(F.col("ingested_at").desc())

silver = (
    good
    .withColumn("rn", F.row_number().over(key))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingested_at")
    .withColumn("country", F.coalesce(F.col("count"), F.lit("Unknown")))
    .withColumn("region", F.coalesce(F.col("admreg"), F.lit("Unknown")))
    .withColumn("doc_type", F.coalesce(F.col("docty"), F.lit("Unknown")))
    .withColumn("language", F.coalesce(F.col("lang"), F.lit("Unknown")))
    .drop("count", "admreg", "docty", "lang")
)

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_docs")
quarantine.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_quarantine")

print("bronze:", bronze.count())
print("silver:", spark.table("silver_docs").count())
print("quarantine:", spark.table("silver_quarantine").count())
display(spark.table("silver_docs").limit(5))

bronze: 4000
silver: 3851
quarantine: 0


id,display_title,majdocty,doc_date,doc_year,country,region,doc_type,language
1000538,Measuring results from climate change programs : performance indicators for GEF,Publications & Research,2000-09-30,2000,Unknown,Unknown,Working Paper (Numbered Series),English
1003171,Climate information and forecasting for development : lessons from the 1997/98 El Nino,Publications & Research,2000-12-31,2000,Unknown,Unknown,Working Paper (Numbered Series),English
10062752,"مدن تتسم بالمرونة تجاه المناخ: دليل الحد من التعرّض لأخطار الكوارث,Climate resilient cities : a primer on reducing vulnerabilities to disasters (Villes résilientes au climat : un livre élémentaire sur la réduction des vulnérabilités aux catastrophes),Ciudades resistentes al clima: una cartilla en la reducción de vulnerabilidades a los desastres",Publications; Publications & Research,2009-02-19,2009,East Asia and Pacific,East Asia and Pacific,Publication,English
10075017,Promoting energy efficiency and renewable energy : GEF climate change projects and impacts,Publications & Research,2000-06-01,2000,World,Other,Global Environment Facility Working Paper,English
10108888,Mexico - Adaptation to Climate Change in the Coastal Wetlands in the Gulf of Mexico Project,Project Documents,2008-12-13,2008,Mexico,Latin America and Caribbean,Integrated Safeguards Data Sheet,English


In [0]:
# документы по годам
spark.sql("""
CREATE OR REPLACE TABLE gold_by_year AS
SELECT doc_year, count(*) AS doc_count
FROM silver_docs
WHERE doc_year IS NOT NULL
GROUP BY doc_year ORDER BY doc_year
""")

# топ стран
spark.sql("""
CREATE OR REPLACE TABLE gold_by_country AS
SELECT country, count(*) AS doc_count
FROM silver_docs
GROUP BY country ORDER BY doc_count DESC
""")

# по регионам
spark.sql("""
CREATE OR REPLACE TABLE gold_by_region AS
SELECT region, count(*) AS doc_count
FROM silver_docs
GROUP BY region ORDER BY doc_count DESC
""")

# по типам документов
spark.sql("""
CREATE OR REPLACE TABLE gold_by_doctype AS
SELECT doc_type, count(*) AS doc_count
FROM silver_docs
GROUP BY doc_type ORDER BY doc_count DESC
""")

# по языкам
spark.sql("""
CREATE OR REPLACE TABLE gold_by_language AS
SELECT language, count(*) AS doc_count
FROM silver_docs
GROUP BY language ORDER BY doc_count DESC
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_country_doctype AS
WITH top_countries AS (
    SELECT country FROM silver_docs
    WHERE country <> 'Unknown'
    GROUP BY country ORDER BY count(*) DESC LIMIT 8
),
top_types AS (
    SELECT doc_type FROM silver_docs
    WHERE doc_type <> 'Unknown'
    GROUP BY doc_type ORDER BY count(*) DESC LIMIT 6
)
SELECT s.country, s.doc_type, count(*) AS doc_count
FROM silver_docs s
JOIN top_countries c ON s.country = c.country
JOIN top_types t ON s.doc_type = t.doc_type
GROUP BY s.country, s.doc_type
ORDER BY s.country, s.doc_type
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_by_region AS
SELECT
  CASE WHEN region LIKE '%;%' THEN 'Multiple regions' ELSE region END AS region,
  count(*) AS doc_count
FROM silver_docs
WHERE region <> 'Unknown'
GROUP BY CASE WHEN region LIKE '%;%' THEN 'Multiple regions' ELSE region END
ORDER BY doc_count DESC
""")

print("gold tables created")
display(spark.table("gold_by_year"))

region,doc_count
Other,720
Latin America and Caribbean,642
Eastern and Southern Africa,639
South Asia,540
East Asia and Pacific,311
Western and Central Africa,284
"Middle East, North Africa, Afghanistan, and Pakistan",260
Africa,164
Europe and Central Asia,161
Multiple regions,58


gold tables created


doc_year,doc_count
1982,1
1985,1
1991,1
1993,1
1994,4
1995,4
1997,5
1998,4
1999,4
2000,8
